**Google Cloud Bigquery con Python**

 La librería de Google Cloud BigQuery para Python es una herramienta poderosa para interactuar con BigQuery, un servicio de almacenamiento y análisis de datos a gran escala.

In [ ]:
!pip install google-cloud-bigquery

In [ ]:
!pip install pandas

In [ ]:
!pip install db_dtypes

In [ ]:
import pandas as pd
from google.cloud import bigquery

In [ ]:
from google.colab import auth
auth.authenticate_user()

**Creación de un cliente BigQuery:**
Para comenzar a trabajar con BigQuery, primero debes crear un cliente. Debes proporcionar las credenciales de autenticación de tu proyecto.

In [ ]:
client = bigquery.Client(project='smart-data-c')

**query():** Envía una consulta a BigQuery y devuelve un objeto google.cloud.bigquery.table.RowIterator que se puede usar para iterar sobre los resultados:

In [ ]:
query = """
    SELECT name, COUNT(*) as count
    FROM `bigquery-public-data.usa_names.usa_1910_2013`
    WHERE state = 'TX'
    GROUP BY name
    ORDER BY count DESC
    LIMIT 10
"""

query_job = client.query(query)
results = query_job.result()
print(list(results))

[Row(('Francis', 208), {'name': 0, 'count': 1}), Row(('Guadalupe', 208), {'name': 0, 'count': 1}), Row(('Jessie', 208), {'name': 0, 'count': 1}), Row(('Marion', 205), {'name': 0, 'count': 1}), Row(('Leslie', 202), {'name': 0, 'count': 1}), Row(('Frankie', 201), {'name': 0, 'count': 1}), Row(('Lee', 196), {'name': 0, 'count': 1}), Row(('Jackie', 195), {'name': 0, 'count': 1}), Row(('Jose', 192), {'name': 0, 'count': 1}), Row(('Johnnie', 191), {'name': 0, 'count': 1})]


In [ ]:
for row in query_job:
    print(f"{row.name}: {row.count}")

Francis: 208
Guadalupe: 208
Jessie: 208
Marion: 205
Leslie: 202
Frankie: 201
Lee: 196
Jackie: 195
Jose: 192
Johnnie: 191


In [ ]:
results_df = query_job.to_dataframe()
results_df

,name,count
0,Francis,208
1,Guadalupe,208
2,Jessie,208
3,Marion,205
4,Leslie,202
5,Frankie,201
6,Lee,196
7,Jackie,195
8,Jose,192
9,Johnnie,191


**create_dataset():** Crea un nuevo conjunto de datos en BigQuery:

In [ ]:
dataset_id = "my_dataset"
dataset = bigquery.Dataset(client.dataset(dataset_id))
dataset.location = "US"
dataset = client.create_dataset(dataset)

**list_datasets():** Lista los conjuntos de datos en un proyecto de BigQuery:

In [ ]:
datasets = list(client.list_datasets())

for dataset in datasets:
    print(dataset.dataset_id)

ecommerce
my_dataset
sd_local_dataset_1


**create_table():** Crea una nueva tabla en BigQuery:

In [ ]:
table_id = "my_table"

schema = [
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("age", "INTEGER", mode="REQUIRED"),
]

table = bigquery.Table(client.dataset(dataset_id).table(table_id), schema=schema)
table = client.create_table(table)

**Exportar resultados de una consulta a un archivo CSV:** Puedes exportar los resultados de una consulta a un archivo CSV.

In [ ]:
query = """
    SELECT *
    FROM `bigquery-public-data.usa_names.usa_1910_2013`
    LIMIT 100
"""
query_df = client.query(query).to_dataframe()
query_df.to_csv("query_results.csv", index=False)

**load_table_from_dataframe():** Carga los datos de un pandas.DataFrame en una tabla de BigQuery:

In [ ]:
data = pd.read_csv("query_results.csv")
dataset_id = 'my_dataset'
table_id = "query_results"

table_ref = client.dataset(dataset_id).table(table_id)

job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.job.WriteDisposition.WRITE_APPEND,
        create_disposition=bigquery.job.CreateDisposition.CREATE_IF_NEEDED,
        autodetect=True
        )

load_job = client.load_table_from_dataframe(data, table_ref, job_config=job_config)

load_job.result()


LoadJob<project=smart-data-c, location=US, id=ce9fd086-8d19-42c1-a997-3facf1a80ba2>

**delete_table():** Elimina una tabla de BigQuery:

In [ ]:
dataset_id = 'my_dataset'
table_id = "query_results"
dataset_ref = client.dataset(dataset_id)
table_ref = dataset_ref.table(table_id)

client.delete_table(table_ref)